# graph

> The correction overlay's graph I/O: targeted (scale-shaped) reads of a committed
> spine via the graph-storage `query` action, construction of Correction /
> CorrectionSession nodes + CORRECTS / SUPERSEDES / DERIVED_FROM / REVIEWED edges,
> the in-core effective-spine projection (layer-0 + applied corrections), and
> commit through the job queue. Hand-rolled (revolution-1) = direct CR-18 spec
> material; append-only on layer-0 (never update/delete a Segment).

In [ ]:
#| default_exp graph

In [ ]:
#| export
import json
import time
from uuid import uuid4
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_context_graph_primitives.locators import locator_from_dict

from cjm_transcript_correction_core.models import (
    Correction, CorrectionSession, CorrectionRelations, SpineSegment,
)

In [ ]:
#| export
# Targeted, scale-shaped read of one document's spine (decomp D13 / CR-18): a
# read-only SELECT over the graph-storage `query` action instead of a
# whole-neighborhood get_context. Cost (pass-2 evidence): this is RAW SQL coupled
# to the sqlite-graph storage schema (`nodes`/`edges` + json_extract) -- the leak
# a typed graph-storage-adapter query would close.
_SEGMENTS_SQL = (
    "SELECT n.id, "
    "json_extract(n.properties, '$.index'), "
    "json_extract(n.properties, '$.text'), "
    "json_extract(n.properties, '$.start_time'), "
    "json_extract(n.properties, '$.end_time'), "
    "n.sources "
    "FROM nodes n "
    "JOIN edges e ON e.source_id = n.id "
    "WHERE e.target_id = ? AND e.relation_type = ? AND n.label = 'Segment' "
    "ORDER BY json_extract(n.properties, '$.index')"
)


def _row_to_spine_segment(row: List[Any]) -> SpineSegment:  # One query row -> SpineSegment
    """Build a SpineSegment from a `query`-action row (first SourceRef carried)."""
    sources = json.loads(row[5]) if row[5] else []
    src = sources[0] if sources else {}
    return SpineSegment(
        id=row[0], index=int(row[1]) if row[1] is not None else -1,
        text=row[2] or "", start_time=row[3], end_time=row[4],
        source_locator=(str(locator_from_dict(src["locator"])) if src.get("locator") else None),
        content_hash=src.get("content_hash"),
    )


async def load_document_segments(
    queue: JobQueue,               # Started job queue
    graph_id: str,                 # Graph-storage capability id
    document_id: str,              # Document node id
    limit: Optional[int] = None,   # Optional page size (LIMIT)
    offset: Optional[int] = None,  # Optional page offset (OFFSET)
) -> List[SpineSegment]:  # Ordered spine segments (by index)
    """Load a document's Segment spine, ordered by index, via the targeted query action."""
    sql = _SEGMENTS_SQL
    params: List[Any] = [document_id, "PART_OF"]
    if limit is not None:
        sql += " LIMIT ?"
        params.append(int(limit))
        if offset is not None:
            sql += " OFFSET ?"
            params.append(int(offset))
    result = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=params)
    return [_row_to_spine_segment(r) for r in ((result or {}).get("rows", []) or [])]


async def load_empty_segments(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> List[SpineSegment]:  # Only the empty-text segments (server-side filtered)
    """Load ONLY a document's empty-text segments (scale-shaped: server-side filter).

    The D14 prune needs ~10% of the spine; pushing the empty-text predicate into
    SQL avoids materializing the whole document (contrast load_document_segments,
    which the neighbour-dependent worklist signals still require). Pass-2: the
    typed query surface should express a "filtered segment stream", not just paging.
    """
    sql = _SEGMENTS_SQL.replace(
        " ORDER BY",
        " AND (json_extract(n.properties, '$.text') = '' "
        "OR json_extract(n.properties, '$.text') IS NULL) ORDER BY",
    )
    result = await submit_and_wait(queue, graph_id, action="query", sql=sql,
                                   params=[document_id, "PART_OF"])
    return [_row_to_spine_segment(r) for r in ((result or {}).get("rows", []) or [])]


async def count_document_segments(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> int:  # Number of Segment nodes PART_OF the document
    """Count a document's segments server-side (no materialization; scale-shaped)."""
    sql = ("SELECT COUNT(*) FROM nodes n JOIN edges e ON e.source_id = n.id "
           "WHERE e.target_id = ? AND e.relation_type = 'PART_OF' AND n.label = 'Segment'")
    result = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[document_id])
    rows = (result or {}).get("rows", []) or []
    return int(rows[0][0]) if rows and rows[0] else 0

In [ ]:
#| export
def _edge(
    source_id: str,                            # Origin node id
    target_id: str,                            # Destination node id
    relation_type: str,                        # Edge relation type
    properties: Optional[Dict[str, Any]] = None,  # Edge properties
) -> Dict[str, Any]:  # Edge wire dict
    """Build an edge wire dict."""
    return {"id": str(uuid4()), "source_id": source_id, "target_id": target_id,
            "relation_type": relation_type, "properties": properties or {}}


def build_correction_node(
    correction_type: str,                  # "text_content" | "punctuation" | "grouping"
    session_id: str,                       # Owning session id
    payload: Dict[str, Any],               # Type-specific payload
    actor: str = "human",                  # Actor
    status: str = "applied",               # Lifecycle status
    canonical_form: Optional[str] = None,  # Optional entity key
    rationale: Optional[str] = None,       # Optional note
) -> Correction:  # The Correction overlay node (not yet committed)
    """Construct a Correction overlay node (pure; commit happens separately)."""
    return Correction(
        correction_type=correction_type, session_id=session_id, payload=payload,
        actor=actor, status=status, canonical_form=canonical_form, rationale=rationale,
    )


def build_prune_correction(
    document_id: str,            # Document being corrected
    pruned: List[SpineSegment],  # Empty layer-0 segments to prune
    session_id: str,             # Owning session id
    actor: str = "human",        # Actor
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """Build one batch grouping Correction that prunes empty segments (D14).

    Non-destructive: layer-0 nodes are NOT deleted; the Correction records the
    pruned ids and DERIVED_FROM edges point at each pruned Segment. The effective
    spine drops them at projection time (reversible by superseding this node).
    """
    payload = {
        "operation": "prune_empty",
        "document_id": document_id,
        "pruned_segment_ids": [s.id for s in pruned],
        "pruned_count": len(pruned),
    }
    node = build_correction_node("grouping", session_id, payload, actor=actor).to_graph_node()
    edges = [_edge(node.id, s.id, CorrectionRelations.DERIVED_FROM) for s in pruned]
    return node.to_dict(), edges

In [ ]:
#| export
async def commit_nodes_edges(
    queue: JobQueue,              # Started job queue
    graph_id: str,                # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts
    edges: List[Dict[str, Any]],  # Edge wire dicts
) -> Dict[str, int]:  # {"nodes": n, "edges": m} created counts
    """Commit nodes then edges via the job queue (queue-routed; decomp D7 telemetry)."""
    nc = ec = 0
    if nodes:
        r = await submit_and_wait(queue, graph_id, action="add_nodes", nodes=nodes)
        nc = (r or {}).get("count", 0)
    if edges:
        r = await submit_and_wait(queue, graph_id, action="add_edges", edges=edges)
        ec = (r or {}).get("count", 0)
    return {"nodes": nc, "edges": ec}


async def start_session(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    scope: List[str],  # Document ids in scope
) -> CorrectionSession:  # The committed CorrectionSession
    """Create + commit a new CorrectionSession node."""
    sess = CorrectionSession(scope=scope)
    await commit_nodes_edges(queue, graph_id, [sess.to_graph_node().to_dict()], [])
    return sess


async def get_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
) -> Optional[Dict[str, Any]]:  # The session node dict, or None
    """Fetch a CorrectionSession node by id (resume/reopen)."""
    r = await submit_and_wait(queue, graph_id, action="get_node", node_id=session_id)
    return (r or {}).get("node", None)


async def set_session_status(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # CorrectionSession node id
    status: str,      # New status ("in_progress" | "completed" | "reopened")
) -> None:
    """Update a session's status + updated_at.

    The ONLY update_node use in this core: a CorrectionSession is OVERLAY metadata
    whose lifecycle is mutable. Layer-0 Segments + Corrections stay append-only.
    """
    await submit_and_wait(queue, graph_id, action="update_node", node_id=session_id,
                          properties={"status": status, "updated_at": time.time()})


async def record_review_markers(
    queue: JobQueue,                   # Started job queue
    graph_id: str,                     # Graph-storage capability id
    session_id: str,                   # Owning session id
    decisions: List[Tuple[str, str]],  # (segment_id, decision) pairs
) -> int:  # Number of REVIEWED edges committed
    """Persist per-(session, segment) review markers as REVIEWED edges."""
    edges = [_edge(session_id, seg_id, CorrectionRelations.REVIEWED, {"decision": dec})
             for seg_id, dec in decisions]
    return (await commit_nodes_edges(queue, graph_id, [], edges))["edges"]


async def load_review_markers(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> Dict[str, str]:  # segment_id -> decision for the session
    """Load a session's review markers (targeted query over REVIEWED edges)."""
    sql = ("SELECT target_id, json_extract(properties, '$.decision') FROM edges "
           "WHERE source_id = ? AND relation_type = 'REVIEWED'")
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[session_id])
    return {row[0]: row[1] for row in ((r or {}).get("rows", []) or [])}

In [ ]:
#| export
async def find_corrections_for_session(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    session_id: str,  # Owning session id
) -> List[Dict[str, Any]]:  # Correction property dicts for the session
    """List corrections recorded in a session (targeted query by session_id property)."""
    sql = ("SELECT id, properties FROM nodes WHERE label = 'Correction' "
           "AND json_extract(properties, '$.session_id') = ?")
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[session_id])
    out = []
    for row in ((r or {}).get("rows", []) or []):
        props = json.loads(row[1]) if row[1] else {}
        props["id"] = row[0]
        out.append(props)
    return out


async def find_prior_corrections_by_hash(
    queue: JobQueue,    # Started job queue
    graph_id: str,      # Graph-storage capability id
    content_hash: str,  # SourceRef content hash to look up
) -> List[Dict[str, Any]]:  # Corrections whose CORRECTS target carried this hash
    """Cross-transcript correction-cache lookup (targeted; the graph IS the lexicon).

    A correction made on identical content in ANY prior transcript is discoverable
    here, keyed by content_hash via a CORRECTS-edge join -- the targeted,
    scale-shaped read the design wants (vs whole-neighborhood get_context).
    """
    sql = ("SELECT c.id, c.properties FROM nodes c "
           "JOIN edges e ON e.source_id = c.id AND e.relation_type = 'CORRECTS' "
           "JOIN nodes s ON s.id = e.target_id, json_each(s.sources) src "
           "WHERE c.label = 'Correction' "
           "AND json_extract(src.value, '$.content_hash') = ?")
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[content_hash])
    out = []
    for row in ((r or {}).get("rows", []) or []):
        props = json.loads(row[1]) if row[1] else {}
        props["id"] = row[0]
        out.append(props)
    return out

In [ ]:
#| export
def project_effective_spine(
    segments: List[SpineSegment],       # Ordered layer-0 spine
    corrections: List[Dict[str, Any]],  # Applied correction property dicts
) -> List[SpineSegment]:  # The effective spine after applying corrections
    """Project the effective spine = layer-0 + applied corrections, resolved in-core.

    Revolution-1 supports the operations built this session:
      - grouping/prune_empty: drop pruned segment ids (re-thread is positional).
      - text_content/replace_text: replace a segment's text by id.
    Superseded corrections are skipped. This hand-rolled projection is direct
    CR-18 spec material (the graph-aware layer should eventually own
    "give me the effective view of this spine").
    """
    pruned_ids = set()
    text_overrides: Dict[str, str] = {}
    for c in corrections:
        if c.get("status") == "superseded":
            continue
        ctype = c.get("correction_type")
        payload = c.get("payload", {}) or {}
        if ctype == "grouping" and payload.get("operation") == "prune_empty":
            pruned_ids.update(payload.get("pruned_segment_ids", []))
        elif ctype == "text_content" and payload.get("operation") == "replace_text":
            tid = payload.get("segment_id")
            if tid is not None:
                text_overrides[tid] = payload.get("new_text", "")
    out: List[SpineSegment] = []
    for s in segments:
        if s.id in pruned_ids:
            continue
        if s.id in text_overrides:
            s = SpineSegment(id=s.id, index=s.index, text=text_overrides[s.id],
                             start_time=s.start_time, end_time=s.end_time,
                             source_locator=s.source_locator, content_hash=s.content_hash)
        out.append(s)
    return out

In [ ]:
#| export
def build_text_correction(
    document_id: str,                      # Document the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text (for the record)
    supersedes_id: Optional[str] = None,   # Prior Correction this one replaces (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key (cross-transcript matching)
    rationale: Optional[str] = None,       # Optional note
) -> Tuple[Dict[str, Any], List[Dict[str, Any]]]:  # (correction node dict, edge dicts)
    """Build a text_content Correction + its CORRECTS (+ optional SUPERSEDES) edges.

    Non-destructive: the layer-0 Segment is unchanged; the correction carries the
    new text in its payload + a CORRECTS edge to the segment. A re-edit adds a
    SUPERSEDES edge (new -> prior) so supersession is graph-native + append-only
    (the prior Correction is never mutated; it is excluded from the effective view
    because it is a SUPERSEDES target). Direct CR-18 spec material.
    """
    payload = {"operation": "replace_text", "document_id": document_id,
               "segment_id": segment_id, "new_text": new_text, "old_text": old_text}
    node = build_correction_node("text_content", session_id, payload, actor=actor,
                                 canonical_form=canonical_form, rationale=rationale).to_graph_node()
    edges = [_edge(node.id, segment_id, CorrectionRelations.CORRECTS)]
    if supersedes_id:
        edges.append(_edge(node.id, supersedes_id, CorrectionRelations.SUPERSEDES))
    return node.to_dict(), edges


async def commit_text_correction(
    queue: JobQueue,                       # Started job queue
    graph_id: str,                         # Graph-storage capability id
    document_id: str,                      # Document the segment belongs to
    segment_id: str,                       # Layer-0 Segment being corrected
    new_text: str,                         # Corrected text
    session_id: str,                       # Owning session id
    old_text: Optional[str] = None,        # Prior effective text
    supersedes_id: Optional[str] = None,   # Prior Correction to supersede (re-edit)
    actor: str = "human",                  # Actor
    canonical_form: Optional[str] = None,  # Optional entity key
) -> str:  # The new Correction node id
    """Commit a text_content correction (node + CORRECTS [+ SUPERSEDES]) + a REVIEWED marker."""
    node, edges = build_text_correction(
        document_id, segment_id, new_text, session_id, old_text=old_text,
        supersedes_id=supersedes_id, actor=actor, canonical_form=canonical_form)
    await commit_nodes_edges(queue, graph_id, [node], edges)
    await record_review_markers(queue, graph_id, session_id, [(segment_id, "corrected")])
    return node["id"]

In [ ]:
#| export
def active_corrections(
    corrections: List[Dict[str, Any]],  # Corrections (e.g. from load_document_corrections)
    superseded_ids: set,                # Ids that are SUPERSEDES targets
) -> List[Dict[str, Any]]:  # Only the effective (non-superseded) corrections
    """Filter to the effective correction set (drop SUPERSEDES targets; append-only supersession)."""
    return [c for c in corrections if c.get("id") not in superseded_ids]


async def _superseded_ids(
    queue: JobQueue,            # Started job queue
    graph_id: str,             # Graph-storage capability id
    correction_ids: List[str],  # Candidate correction ids
) -> set:  # Subset that are SUPERSEDES targets
    """Return which of the given corrections have been superseded (are SUPERSEDES targets)."""
    if not correction_ids:
        return set()
    ph = ",".join("?" for _ in correction_ids)
    sql = f"SELECT DISTINCT target_id FROM edges WHERE relation_type = 'SUPERSEDES' AND target_id IN ({ph})"
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=correction_ids)
    return {row[0] for row in ((r or {}).get("rows", []) or [])}


async def load_document_corrections(
    queue: JobQueue,   # Started job queue
    graph_id: str,     # Graph-storage capability id
    document_id: str,  # Document node id
) -> Tuple[List[Dict[str, Any]], set]:  # (all corrections for the doc, superseded id set)
    """Load every Correction targeting a document (across sessions) + the superseded-id set.

    Document-scoped (corrections carry payload.document_id) so the effective view is
    a property of the document, not one session -- the persistence/resume/reopen
    requirement. Append-only: supersession is read from SUPERSEDES edges, never a
    status mutation.
    """
    sql = ("SELECT id, properties FROM nodes WHERE label = 'Correction' "
           "AND json_extract(properties, '$.payload.document_id') = ?")
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[document_id])
    corrections = []
    for row in ((r or {}).get("rows", []) or []):
        props = json.loads(row[1]) if row[1] else {}
        props["id"] = row[0]
        corrections.append(props)
    superseded = await _superseded_ids(queue, graph_id, [c["id"] for c in corrections])
    return corrections, superseded


async def find_active_text_correction(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability id
    segment_id: str,  # Segment to look up
) -> Optional[Dict[str, Any]]:  # The current non-superseded text correction, or None
    """Find the active (non-superseded) text correction on a segment, across sessions (latest wins).

    Cross-session lookup is what makes a re-edit in a LATER session supersede an
    earlier correction (the resume/reopen requirement). The graph is the cache.
    """
    sql = ("SELECT id, properties FROM nodes WHERE label = 'Correction' "
           "AND json_extract(properties, '$.correction_type') = 'text_content' "
           "AND json_extract(properties, '$.payload.segment_id') = ?")
    r = await submit_and_wait(queue, graph_id, action="query", sql=sql, params=[segment_id])
    cands = []
    for row in ((r or {}).get("rows", []) or []):
        props = json.loads(row[1]) if row[1] else {}
        props["id"] = row[0]
        cands.append(props)
    if not cands:
        return None
    superseded = await _superseded_ids(queue, graph_id, [c["id"] for c in cands])
    active = [c for c in cands if c["id"] not in superseded]
    return max(active, key=lambda c: c.get("created_at", 0.0)) if active else None

In [ ]:
# graph pure-logic checks (no runtime)
segs = [
    SpineSegment(id="a", index=0, text="hello"),
    SpineSegment(id="b", index=1, text=""),
    SpineSegment(id="c", index=2, text="world"),
]
empties = [s for s in segs if s.is_empty]
node, edges = build_prune_correction("doc1", empties, session_id="s1")
assert node["label"] == "Correction"
assert node["properties"]["payload"]["pruned_segment_ids"] == ["b"]
assert len(edges) == 1 and edges[0]["relation_type"] == "DERIVED_FROM"
assert edges[0]["target_id"] == "b" and edges[0]["source_id"] == node["id"]

eff = project_effective_spine(segs, [node["properties"]])
assert [s.id for s in eff] == ["a", "c"]               # prune drops the empty segment

repl = {"correction_type": "text_content", "status": "applied",
        "payload": {"operation": "replace_text", "segment_id": "a", "new_text": "HELLO"}}
assert project_effective_spine(segs, [repl])[0].text == "HELLO"

sup = dict(node["properties"]); sup["status"] = "superseded"
assert [s.id for s in project_effective_spine(segs, [sup])] == ["a", "b", "c"]  # superseded ignored
print("graph pure checks OK")

graph pure checks OK


In [ ]:
# graph text-correction pure checks (no runtime)
tn, te = build_text_correction("doc1", "segX", "fixed text", session_id="s1", supersedes_id="prevCorr")
assert tn["properties"]["correction_type"] == "text_content"
assert tn["properties"]["payload"]["segment_id"] == "segX"
assert tn["properties"]["payload"]["new_text"] == "fixed text"
assert tn["properties"]["payload"]["document_id"] == "doc1"
assert {e["relation_type"] for e in te} == {"CORRECTS", "SUPERSEDES"}
corr_edge = next(e for e in te if e["relation_type"] == "CORRECTS")
assert corr_edge["target_id"] == "segX" and corr_edge["source_id"] == tn["id"]
assert next(e for e in te if e["relation_type"] == "SUPERSEDES")["target_id"] == "prevCorr"
assert [c["id"] for c in active_corrections([{"id": "a"}, {"id": "b"}, {"id": "c"}], {"b"})] == ["a", "c"]
print("graph text-correction checks OK")